To create an ``CPP`` object, you just need to provide a valid ``df_parts`` DataFrame:

In [1]:
import aaanalysis as aa
df_seq = aa.load_dataset(name="DOM_GSEC", n=50)
sf = aa.SequenceFeature()
df_parts = sf.get_df_parts(df_seq=df_seq)
# Create CPP object
cpp = aa.CPP(df_parts=df_parts)

You can adjust **Parts**, **Splits**, and **Scales** as follows:

In [2]:
df_parts = sf.get_df_parts(df_seq=df_seq, list_parts=["tmd_jmd"])
split_kws = sf.get_split_kws(split_types=["Segment"], n_split_max=1)
df_scales = aa.load_scales()
scales = list(df_scales)[0:10]
# Create CPP object for Segments over the complete TMD-JMD with 10 first scales 
cpp = aa.CPP(df_parts=df_parts, split_kws=split_kws, df_scales=df_scales[scales])

The ``CPP`` constructor also accepts ``df_cat`` (the scale-category table used by the redundancy filter and plotting; loaded automatically when omitted), ``accept_gaps`` (tolerate gap symbols in the sequence parts), ``random_state`` (reproducibility), and ``verbose`` (progress messages):

In [3]:
labels = df_seq["label"].to_list()
df_scales = aa.load_scales(top_explain_n=20)
df_cat = aa.load_scales(name="scales_cat", top_explain_n=20)
cpp = aa.CPP(df_parts=df_parts, df_scales=df_scales, df_cat=df_cat,
             accept_gaps=False, random_state=0, verbose=False)
df_feat = cpp.run(labels=labels, n_filter=10)
aa.display_df(df_feat, n_rows=5, show_shape=True)

/Users/stephanbreimann/Programming/1Packages/wt-368-bootstrap/aaanalysis/feature_engineering/_backend/cpp_run.py:164: UserWarning: CPP is using the Python kernel fallback — the compiled Cython extension is not available in this install. Output is bit-exact with the Cython path but ~2x slower. Reinstall via `pip install --force-reinstall aaanalysis` to fetch a prebuilt wheel.
  warnings.warn(


DataFrame shape: (10, 13)


,feature,category,subcategory,scale_name,scale_description,abs_auc,abs_mean_dif,mean_dif,std_test,std_ref,p_val_mann_whitney,p_val_fdr_bh,positions
1,"TMD_JMD-Segment...,15)-FAUJ880104",Shape,Side chain length,Steric parameter,"STERIMOL length...e et al., 1988)",0.382000,0.264000,0.264000,0.156000,0.156000,0.000000,0.000000,"33,34"
2,"TMD_JMD-Segment...8,9)-HUTJ700103",Energy,Entropy,Entropy,"Entropy of form...Hutchens, 1970)",0.378000,0.212000,0.212000,0.124000,0.135000,0.000000,0.000000,"32,33,34,35"
3,"TMD_JMD-Pattern...,14)-CRAJ730103",Conformation,β-turn,β-turn,"Normalized freq...d et al., 1973)",0.376000,0.281000,-0.281000,0.159000,0.180000,0.000000,0.000000,"27,31"
4,"TMD_JMD-Segment...,13)-FAUJ880104",Shape,Side chain length,Steric parameter,"STERIMOL length...e et al., 1988)",0.364000,0.275000,0.275000,0.172000,0.177000,0.000000,0.000000,"31,32,33"
5,"TMD_JMD-Pattern...,14)-QIAN880107",Conformation,α-helix,α-helix (middle),"Weights for alp...ejnowski, 1988)",0.359000,0.199000,0.199000,0.112000,0.145000,0.000000,0.000000,"27,31"


For a more **stable, reproducible** feature list, enable bootstrap stability selection on the constructor. Each of ``n_bootstrap`` rounds resamples the data (``resample`` chooses which class group is resampled: ``"reference"`` fixes the test group, or ``"both"`` / ``"test"``; ``bootstrap_frac`` is the per-group draw size), re-selects features, and the ``n_freq`` most frequently selected are kept. Those candidates are then re-evaluated and filtered on the full dataset, and ``df_feat`` gains a ``selection_frequency`` column (fraction of rounds each feature was selected). The mode applies to :meth:`run` and :meth:`run_num` alike; the default ``n_bootstrap=0`` disables it:

In [4]:
cpp = aa.CPP(df_parts=df_parts, df_scales=df_scales, df_cat=df_cat, random_state=0, verbose=False,
             n_bootstrap=20, resample="reference", bootstrap_frac=0.8, n_freq=50)
df_feat = cpp.run(labels=labels, n_filter=10)
aa.display_df(df_feat, n_rows=5, show_shape=True)

DataFrame shape: (10, 14)


,feature,category,subcategory,scale_name,scale_description,abs_auc,abs_mean_dif,mean_dif,std_test,std_ref,p_val_mann_whitney,p_val_fdr_bh,positions,selection_frequency
1,"TMD_JMD-Segment...,15)-FAUJ880104",Shape,Side chain length,Steric parameter,"STERIMOL length...e et al., 1988)",0.382000,0.264000,0.264000,0.156000,0.156000,0.000000,0.000000,"33,34",0.500000
2,"TMD_JMD-Segment...8,9)-HUTJ700103",Energy,Entropy,Entropy,"Entropy of form...Hutchens, 1970)",0.378000,0.212000,0.212000,0.124000,0.135000,0.000000,0.000000,"32,33,34,35",0.450000
3,"TMD_JMD-Pattern...,14)-CRAJ730103",Conformation,β-turn,β-turn,"Normalized freq...d et al., 1973)",0.376000,0.281000,-0.281000,0.159000,0.180000,0.000000,0.000000,"27,31",0.750000
4,"TMD_JMD-Segment...,13)-FAUJ880104",Shape,Side chain length,Steric parameter,"STERIMOL length...e et al., 1988)",0.364000,0.275000,0.275000,0.172000,0.177000,0.000000,0.000000,"31,32,33",0.300000
5,"TMD_JMD-Pattern...,14)-QIAN880107",Conformation,α-helix,α-helix (middle),"Weights for alp...ejnowski, 1988)",0.359000,0.199000,0.199000,0.112000,0.145000,0.000000,0.000000,"27,31",0.150000
